*PART 1*

In [1]:
# PART 1
# import necessary libraries
import numpy as np
import pandas as pd

In [2]:
data=pd.read_csv("diabetes.csv")

In [4]:
data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
data.isna().sum() # finding if any empty row exists

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [7]:
data.describe() # statistical summary

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [11]:
# count for number of patients with 0 pregnancies
(data['Pregnancies']==0).value_counts()

,count
Pregnancies,
False,657
True,111


In [12]:
# count of patients with diabetes
(data['Outcome']==1).value_counts()

,count
Outcome,
False,500
True,268


In [14]:
# 3. Separate the dataset into:
# a. Features (X)
# b. Target Variable (y)
x = data.drop('Outcome',axis=1)
y = data['Outcome']

In [15]:
# 4. Perform a Train-Test Split.
# splitting the data into training and testing set
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.80)

In [16]:
# 5 Perform appropriate preprocessing, if required:
# a. Handle missing values
# b. Encode categorical variables, if any
# c. Scale/standardize numerical features where appropriate
#===========================================================#
# There are 0 missing values but there are values with value 0 which can be modified with a median value for example BMI
# Columns where 0 can be replaced with a median value : Glucose , BloodPressure , SkinThickness , Insulin , BMI
# treat zero values in those five medical columns as missing values
from sklearn.impute import SimpleImputer

# Columns where 0 means missing / invalid data
columns_to_impute=[
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI'
]

# Replace 0 with NaN in the feature datasets
x_train[columns_to_impute] = x_train[columns_to_impute].replace(0,np.nan)
x_test[columns_to_impute] = x_test[columns_to_impute].replace(0,np.nan)

# Create median imputer
imputer = SimpleImputer(strategy='median')

# Fit on training data and transform training data
x_train[columns_to_impute] = imputer.fit_transform(x_train[columns_to_impute])

# Transform testing data using training medians
x_test[columns_to_impute] = imputer.fit_transform(x_test[columns_to_impute])

In [17]:
# Standardization
from sklearn.preprocessing import StandardScaler

# Create Scaler
scaler = StandardScaler()

# fit and transform training data
x_train = scaler.fit_transform(x_train)

# transform testing data using testing statistics
x_test = scaler.transform(x_test)

#**we scale only the independent variable x**

In [18]:
# Verify preprocessing
np.isnan(x_train).sum()

np.int64(0)

In [19]:
# Verify preprocessing
np.isnan(x_test).sum()

np.int64(0)

*Part 2 — Machine Learning Model Building*

*Build a suitable classification model using Scikit-learn.*

In [37]:
# import required libraries

# Machine learning algorithm
from sklearn.linear_model import LogisticRegression

# Model evaluation (used in the next stage)
from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)

In [21]:
# create logistic regression model
model = LogisticRegression()

PART 3 - Model Training

*Train the Machine Learning model using the prepared training dataset.*

In [23]:
# train the model
# Fit the selected model using the training dataset.
model.fit(x_train,y_train)

LogisticRegression()

In [26]:
# Generate predictions on the training dataset.
y_train_predictions = model.predict(x_train)

In [28]:
# Create result of actual and predicted train outcomes
results = pd.DataFrame({
    'Actual Outcome': y_train,
    'Predicted Outcome': y_train_predictions
})

In [29]:
# result
results.head(10)

,Actual Outcome,Predicted Outcome
714,0,0
622,0,1
105,0,0
629,0,0
468,1,0
322,1,0
297,0,0
633,0,0
490,0,0
549,0,1


In [31]:
# Calculate training accuracy
accuracy_score(y_train, y_train_predictions)

0.7882736156351792

In [43]:
# Classification Report for train
classification_report(y_train,y_train_predictions,target_names=['No Diabetes', 'Diabetes'])

'              precision    recall  f1-score   support\n\n No Diabetes       0.81      0.89      0.85       408\n    Diabetes       0.73      0.58      0.65       206\n\n    accuracy                           0.79       614\n   macro avg       0.77      0.74      0.75       614\nweighted avg       0.78      0.79      0.78       614\n'

In [44]:
# Confusion Matrix for train
#[
#   [ TN(Correctly predicted No Diabetes)   FP(Incorrectly predicted diabetes) ],
#   [ FN(Missed diabetes cases)             TP(Correctly predicted diabetes)  ]
#]
confusion_matrix(y_train,y_train_predictions)

array([[364,  44],
       [ 86, 120]])

In [32]:
# Generate prediction on testing data
y_test_predictions = model.predict(x_test)

In [34]:
# Create result of actual and predicted test outcomes
test_results = pd.DataFrame({
    'Actual Outcome': y_test,
    'Predicted Outcome': y_test_predictions
})

In [35]:
test_results.head(10)

,Actual Outcome,Predicted Outcome
33,0,0
143,1,0
501,0,0
54,0,1
99,1,1
382,0,0
483,0,0
401,0,0
519,0,0
663,1,1


In [36]:
# calculating testing accuracy
accuracy_score(y_test,y_test_predictions)

0.7077922077922078

In [41]:
# Classification Report for test
classification_report(y_test,y_test_predictions,target_names=['No Diabetes', 'Diabetes'])

'              precision    recall  f1-score   support\n\n No Diabetes       0.72      0.84      0.77        92\n    Diabetes       0.68      0.52      0.59        62\n\n    accuracy                           0.71       154\n   macro avg       0.70      0.68      0.68       154\nweighted avg       0.70      0.71      0.70       154\n'

In [40]:
# Confusion Matrix
#[
#   [ TN(Correctly predicted No Diabetes)   FP(Incorrectly predicted diabetes) ],
#   [ FN(Missed diabetes cases)             TP(Correctly predicted diabetes)  ]
#]
confusion_matrix(y_test,y_test_predictions)

array([[77, 15],
       [30, 32]])